# Paper 3 reproducibility notebook

Public, sanitized notebook for release `v1.4-paper3`. It reproduces metrics calculable from public aggregate tables and validates restricted-workflow aggregates without exposing row-level records.


In [ ]:
from csv import DictReader
from pathlib import Path
ROOT=Path.cwd(); DATA=ROOT/'paper3_metodos'/'data'
if not DATA.exists(): DATA=Path('..')/'data'
def read_csv(name):
    with (DATA/name).open(newline='',encoding='utf-8') as fh: return list(DictReader(fh))
def pct(n,d): return round(n/d*100,2)
summary={r['metric']:r for r in read_csv('validation_summary.csv')}
fields=read_csv('concordance_by_field.csv'); years=read_csv('concordance_by_year.csv')
sens=read_csv('sensitivity_summary.csv'); idem=read_csv('idem_resolution_summary.csv')
review=read_csv('reviewer_agreement_summary.csv')[0]
review_fields={r['field']:r for r in read_csv('reviewer_agreement_by_field.csv')}
ctrl={r['reviewer']:r for r in read_csv('independent_control_summary.csv')}
boot=read_csv('bootstrap_summary.csv')[0]


In [ ]:
comparisons=sum(int(r['comparisons']) for r in fields); matches=sum(int(r['matches']) for r in fields)
assert comparisons==1980 and matches==1867 and comparisons-matches==113 and pct(matches,comparisons)==94.29
by_year={int(r['year']):r for r in years}; by_field={r['field']:r for r in fields}
assert int(by_year[1912]['disagreements'])==59 and float(by_year[1912]['agreement_percent'])==82.12
assert int(by_year[1914]['disagreements'])==24 and int(by_field['padre_tutor']['disagreements'])==31
corpus=sum(int(r['corpus_records']) for r in years)
weighted=sum((int(r['matches'])/int(r['comparisons']))*(int(r['corpus_records'])/corpus) for r in years)
assert round(weighted*100,2)==94.41
print({'assisted_audit_agreement':94.29,'weighted_agreement':94.41})


In [ ]:
s={r['scenario']:r for r in sens}['excluding_empty_empty']
assert int(s['excluded_comparisons'])==295 and int(s['comparisons'])==1685 and int(s['matches'])==1572
assert pct(1572,1685)==93.29


In [ ]:
assert int(review['records'])==60 and int(review['comparisons'])==1080
assert int(review['exact_matches'])==1075 and int(review['disagreements'])==5
assert pct(1075,1080)==99.54
assert int(review_fields['nombre_alumno']['disagreements'])==4 and float(review_fields['nombre_alumno']['agreement_percent'])==93.33
assert int(review_fields['domicilio']['disagreements'])==1 and float(review_fields['domicilio']['agreement_percent'])==98.33
for field,row in review_fields.items():
    if field not in {'nombre_alumno','domicilio'}: assert int(row['disagreements'])==0
print({'interreviewer_agreement':99.54,'disagreements':5})


In [ ]:
for reviewer in ('A','B'):
    r=ctrl[reviewer]
    assert int(r['records'])==60 and int(r['comparisons'])==660 and int(r['matches'])==594
    assert float(r['agreement_percent'])==90.00 and int(r['informative_comparisons'])==559 and int(r['informative_matches'])==493
    assert float(r['informative_agreement_percent'])==88.19
assert float(ctrl['A']['weighted_agreement_percent'])==89.97
assert (float(ctrl['A']['ci95_lower_percent']),float(ctrl['A']['ci95_upper_percent']))==(86.45,93.29)
assert float(ctrl['B']['weighted_agreement_percent'])==89.96
assert (float(ctrl['B']['ci95_lower_percent']),float(ctrl['B']['ci95_upper_percent']))==(86.44,93.26)
print({'baseline_vs_A':90.0,'baseline_vs_B':90.0})


In [ ]:
i=idem[0]
assert int(i['marks_evaluated'])==398 and int(i['resolved_correctly'])==397 and int(i['unresolved_or_incorrect'])==1
assert boot['measure']=='annual_composition_weighted_agreement' and float(boot['estimate_percent'])==94.41
assert float(boot['ci95_lower_percent'])==92.70 and float(boot['ci95_upper_percent'])==95.98 and int(boot['replicates'])==5000


## Interpretation

The 180-record result is an assisted human-in-the-loop audit. The 60-record control was independently reviewed without prefill by two reviewers; because five human disagreements remain, baseline agreement is reported separately against reviewer A and reviewer B rather than collapsing them into a single human reference. These metrics are not CER, WER, or isolated HTR-engine accuracy.
